In [1]:
#Data Loading
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("esp32_Datasheet_en.pdf")
pdfData = loader.load()
print(pdfData[0])

C:\Users\Shreyash\AppData\Local\Temp\ipykernel_8304\2921036784.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Shreyash\OneDrive\Desktop\genaiproj\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


page_content='ESP32 Series
Datasheet Version 5.2
2.4 GHz Wi-Fi + Bluetooth ® + Bluetooth LE SoC
Including:
ESP32-D0WD-V3
ESP32-U4WDH
ESP32-S0WD – Not Recommended for New Designs (NRND)
ESP32-D0WD – Not Recommended for New Designs (NRND)
ESP32-D0WDQ6 – Not Recommended for New Designs (NRND)
ESP32-D0WDQ6-V3 – Not Recommended for New Designs (NRND)
ESP32-D0WDR2-V3 – End of Life (EOL), upgraded to ESP32-D0WDRH2-V3
www.espressif .com' metadata={'producer': 'xdvipdfmx (20240407)', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-28T10:45:12+00:00', 'source': 'esp32_Datasheet_en.pdf', 'total_pages': 78, 'page': 0, 'page_label': '1'}


In [2]:
# Splitting the PDF into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

textSplitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

finalDocument = textSplitter.split_documents(pdfData)

print("Total Chunks:", len(finalDocument))
print(finalDocument[0])

Total Chunks: 315
page_content='ESP32 Series
Datasheet Version 5.2
2.4 GHz Wi-Fi + Bluetooth ® + Bluetooth LE SoC
Including:
ESP32-D0WD-V3
ESP32-U4WDH
ESP32-S0WD – Not Recommended for New Designs (NRND)
ESP32-D0WD – Not Recommended for New Designs (NRND)
ESP32-D0WDQ6 – Not Recommended for New Designs (NRND)
ESP32-D0WDQ6-V3 – Not Recommended for New Designs (NRND)
ESP32-D0WDR2-V3 – End of Life (EOL), upgraded to ESP32-D0WDRH2-V3
www.espressif .com' metadata={'producer': 'xdvipdfmx (20240407)', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-28T10:45:12+00:00', 'source': 'esp32_Datasheet_en.pdf', 'total_pages': 78, 'page': 0, 'page_label': '1'}


In [3]:
# Data Embedding

from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3872.30it/s]


Embedding model loaded successfully!


In [4]:
# Vector Storage using FAISS

from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(
    finalDocument,
    embedding
)

print("Vector database created successfully!")
print(db)

Vector database created successfully!


In [5]:
# Retrieval (Similarity Search)

query = input("Ask a question about ESP32: ")

docs = db.similarity_search(query, k=3)

print("\nTop 3 Relevant Chunks:\n")

for i, doc in enumerate(docs, start=1):
    print(f"----------- Result {i} -----------")
    print(doc.page_content)
    print()


Top 3 Relevant Chunks:

----------- Result 1 -----------
Product Overview
ESP32 is a single 2.4 GHz Wi-Fi-and-Bluetooth combo chip designed with the TSMC low-power 40 nm
technology . It is designed to achieve the best power and RF performance, showing robustness, versatility and
reliability in a wide variety of applications and power scenarios.
For details on part numbers and ordering information, please refer to Section 1 ESP32 Series Comparison . For
details on chip revisions, please refer to ESP32 Chip Revision v3.0 User Guide and

----------- Result 2 -----------
Contents
Note:
Check the link or the QR code to make sure that you use the latest version of this document:
https://www.espressif.com/documentation/esp32_datasheet_en.pdf
Contents
Product Overview 2
Features 3
Applications 5
1 ESP32 Series Comparison 11
1.1 Nomenclature 11
1.2 Comparison 11
2 Pins 12
2.1 Pin Layout 12
2.2 Pin Overview 14
2.3 IO Pins 17
2.3.1 Restrictions for GPIOs and RTC_GPIOs 17
2.4 Analog Pins 17
2.5 P

In [7]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [8]:
llm = Ollama(model="gemma2:2b")

prompt = ChatPromptTemplate.from_template("""
You are an expert ESP32 assistant.

Answer the user's question only using the context below.

Context:
{context}

Question:
{question}
""")

output_parser = StrOutputParser()

C:\Users\Shreyash\AppData\Local\Temp\ipykernel_8304\1667946405.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma2:2b")


In [9]:
query = input("Ask your question about ESP32: ")

docs = db.similarity_search(query, k=3)

context = "\n\n".join([doc.page_content for doc in docs])

chain = prompt | llm | output_parser

response = chain.invoke({
    "context": context,
    "question": query
})

print("\nAnswer:\n")
print(response)


Answer:

You'll find information about ESP32 applications on the ESP32 Series Apps page.  Please check it out for more details! 
[Link to ESP32 applications] 

